### 1. midi 파일 전처리

In [1]:
from util import *
from process import *
from professor import *

file_name = "Ryuichi_Sakamoto_-_hibari.mid"
adjusted_notes, tempo = adjust_to_eighth_note(file_name)

adn_1 = adjusted_notes[:2006]
adn_2 = adjusted_notes[2006:]

adn_1_real = adn_1[:-59]
adn_2_real = adn_2[59:]

adn_1_chord = label_active_chord_by_onset(adn_1_real) # 33*59
adn_2_chord = label_active_chord_by_onset(adn_2_real) # 32*59

adn_i = get_ready_with_lags(adn_1_chord, adn_2_chord)

1th instrument ending : index 2006
2th instrument ending : index 3953


c:\Users\82104\Developments\.venv\lib\site-packages\pretty_midi\pretty_midi.py:100: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(


* 곡에서 반복되는 단위를 추출하고(module_notes), 각 시점에서 활성화된 음들을 모아(active_module) 화음 단위로 정제합니다.
* 화음에 대해 3가지 방식으로 딕셔너리를 만듭니다.

In [2]:
from process import (group_notes_with_duration_, notes_label_n_counts, 
                  chord_label_to_note_labels, transform_dict, chord_label_dict)

module_notes = adn_1_real[:59]
# module_chord = adn_1_chord[:32]
active_module = group_notes_with_duration_(module_notes)

notes_label, notes_counts = notes_label_n_counts(module_notes)
chord_label = chord_label_dict(active_module)

# notes = (pitch, duration)
notes_dict = chord_label_to_note_labels(chord_label, notes_label)
# pitches_dict = transform_dict(chord_label)
# pitch_classes_dict = transform_dict(chord_label, project=True)

notes_dict['name'] = 'notes'
# pitches_dict['name'] = 'pitches'
# pitch_classes_dict['name'] = 'pitch_classes'

* 총 17개의 화음이 등장하고, 각 instrumental의 솔로 구간이 있어 17 choose 2 + 17 = 153개의 조합이 등장할 거란 예상처럼 0 ~ 152까지의 value가 발견됩니다.

In [3]:
from process import simul_chord_lists

simul_chords_key = simul_chord_lists(adn_i[1][-1], adn_i[2][-1])

* 곡의 중반을 중심으로 대칭적으로 구성된 음악입니다.

In [4]:
# import matplotlib.pyplot as plt
# from process import label_simul_chords_combi


# simul_chord_comb_dict = label_simul_chords_combi(simul_chords_key)
# # for item_set, label in simul_chord_comb_dict.items():
# #     print(f"{set(item_set)}: {label}")

# # simul_chord_comb_dict
# labeled_list = []
# for sublist in simul_chords_key:
#     key = frozenset(sublist)  # 현재 sublist를 frozenset으로 변환
#     label = simul_chord_comb_dict[key]  # 해당 frozenset을 키로 사용하여 label 가져오기
#     labeled_list.append(label)  # label을 labeled_list에 추가

# # x축 데이터 (index) 생성
# x = list(range(len(labeled_list)))

# # 그래프 생성
# plt.plot(x, labeled_list, linestyle='-')  # marker와 linestyle을 지정하여 꺾은선 그래프 생성

# # marker='o',

# # 그래프 제목 및 축 레이블 설정
# plt.title("Labeled List Plot")
# plt.xlabel("time")
# plt.ylabel("Label")

# # 그리드 추가 (선택 사항)
# plt.grid(True)

# # 그래프 출력
# plt.show()

# from collections import Counter

# # Counter 객체 생성
# label_counts = Counter(labeled_list)

# frequent_labels = []
# often_labels = []
# for label, count in label_counts.items():
#     if count >= 20 :
#         # print(f"Label {label}: {count}번")
#         frequent_labels.append(label)
#     elif count >= 8 : # 8 아니면 10
#         often_labels.append(label)

# # frequent_chord_comb = [key for key, value in simul_chord_comb_dict.items() if value in frequent_labels]
# often_chord_comb = [key for key, value in simul_chord_comb_dict.items() if value in often_labels]

# # label 빈도를 저장할 딕셔너리 초기화
# label_counts = {i: 0 for i in range(17)}  # 0부터 16까지 label 초기화

# # label 빈도 계산
# for chord_comb in often_chord_comb:
#     for label in chord_comb:
#         # if 0 <= label <= 16:
#         label_counts[label] += 1

# # 결과 출력
# print("Label Counts:")
# for label, count in label_counts.items():
#     print(f"Label {label}: {count}번")

In [5]:
# labeled_list[554] 

* generateBarcode 실행결과 (w.r.t. listofDimension)
  * [1, 2, 3] : 33.2s
  * [1] : 0.1s
  * [2] : 2.2s 
  * [3] : 36.7s 
  * [1, 2] : 2.1s
  * [2, 3] : 33.6s

* 아래 코드를 수행하면 
* intra_weights와 inter_weight를 더해서 얻어진 timeflow_weight에 대해 UTM을 구하고 refine하는 것과
* intra_weights, inter_weight 각각에 대해 UTM을 구하고 refine한 뒤 더하는 연산이 commute함을 알 수 있다.

In [6]:
# from util import check_commutivity
# from util import get_UTMconnected, refine_connectedness, get_distance_matrix_from, get_LTMpart_of
# from util import get_outta_reach, get_chords_inter_connected, is_distance_matrix_from

# inter_lag = 1
# inter_weight = get_chords_inter_connected(adn_i[1][inter_lag], adn_i[2][inter_lag], lag=inter_lag)
# outta_reach_t = get_outta_reach(inter_weight, power = power)

# timeflow_weight = intra_weights + inter_weight 

# """ 여기서부터  """
# # lower triangle에 있는 비대각원소를 UT로 옮깁니다(더한 뒤 0으로 만듭니다) 
# weight_UTM = get_UTMconnected(timeflow_weight)

# # 화음 단위에서 notes(pitch, length) | pitch | pitch_classes 기준으로 weight를 refine합니다
# weight_UTM = refine_connectedness(weight_UTM, refine_dict)

# # 역수 변환
# distance_UTM = get_distance_matrix_from(weight_UTM, outta_reach_t)

# distance_df = get_LTMpart_of(distance_UTM)

# """ 여기까지 is_distance_matrix_from 함수 내부 동작 """

# check_commutivity(inter_weight, intra_weights, refine_dict, outta_reach_t, distance_df)

### timeflow cycle / void 탐색


In [7]:
from util import (search_timeflow_homology, get_rBD_groupedBy_homol, plot_homol_BirthDeath_over_rate, 
                  split_cycles_by_consecutive, homol_rBD_to_pkl)

In [8]:
# inter_lag = 4
# start = 0.0
# end = 1.5
# power_t = -2
# dim_t = 2

In [9]:
# h2_profile_t, outta_reach_t = search_timeflow_homology(adn_i, inter_lag = inter_lag, refine_dict = notes_dict,
#                                                     dimension = dim_t, 
#                                                     rate_start = start, rate_end = end, power = power_t)

# h2_persistence_t = get_rBD_groupedBy_homol(h2_profile_t, dim = dim_t)
# print("Total number of homol: ", len(h2_persistence_t))

# non_consecutive_h2_t, consecutive_h2_t = split_cycles_by_consecutive(h2_persistence_t, out_of_reach = outta_reach_t, power = power_t)
# # plot_homol_BirthDeath_over_rate(non_consecutive_h2_t, subplot_in_a_row = 3, power = power_t)
# plot_homol_BirthDeath_over_rate(consecutive_h2_t, subplot_in_a_row = 4, power = power_t)

# if len(non_consecutive_h2_t) >= 1:
#     h2_persistence_t_updated = {**non_consecutive_h2_t, **consecutive_h2_t}
# else :
#     h2_persistence_t_updated = consecutive_h2_t

# # 데이터프레임 변환 및 pkl파일 내보내기
# temp_t_n_5 = homol_rBD_to_pkl(h2_persistence_t_updated, notes_dict, inter_lag, dim_t, start, end, power_t, type = 't')

### s(simul), t(timeflow), c(complex)에서 탐색한 결과 분석

In [34]:
from util import homol_rBD_from_pkl, catch_intersection

dim = 2
h2_t_persistence_1 = homol_rBD_from_pkl('h2_rBD_t_notes1_1e-2_0.0~1.5.pkl', dir="./pickle")
h2_t_persistence_2 = homol_rBD_from_pkl('h2_rBD_t_notes2_1e-2_0.0~1.5.pkl', dir="./pickle")
h2_t_persistence_3 = homol_rBD_from_pkl('h2_rBD_t_notes3_1e-2_0.0~1.5.pkl', dir="./pickle")
h2_t_persistence_4 = homol_rBD_from_pkl('h2_rBD_t_notes4_1e-2_0.0~1.5.pkl', dir="./pickle")
h2_t_2_1, h2_t_2_2, h2_t_2_3, h2_t_2_4, h2_t_2_all, h2_t_2_u = catch_intersection(h2_t_persistence_1, h2_t_persistence_2, h2_t_persistence_3, h2_t_persistence_4, dim)

Lag 1, 2, 3, 4 모두에서 발견되는 void : {frozenset({(16, 13, 5), (18, 14, 5), (5, 11, 18), (18, 11, 9), (9, 13, 16), (16, 14, 9), (9, 11, 13), (9, 14, 18), (13, 11, 5), (5, 14, 16)}), frozenset({(5, 14, 18), (18, 14, 9), (9, 11, 18), (13, 11, 9), (18, 11, 5), (5, 13, 16), (5, 11, 13), (16, 14, 5), (16, 13, 9), (9, 14, 16)}), frozenset({(5, 11, 18), (18, 11, 9), (19, 18, 9), (3, 5, 19), (19, 9, 3), (3, 9, 11), (5, 18, 19), (11, 5, 3)}), frozenset({(15, 9, 3), (15, 5, 0), (5, 11, 18), (18, 15, 0), (18, 11, 9), (9, 15, 18), (3, 5, 15), (3, 9, 11), (0, 5, 18), (11, 5, 3)}), frozenset({(3, 9, 19), (18, 11, 5), (9, 18, 19), (19, 5, 3), (19, 18, 5), (3, 5, 11), (11, 9, 3), (9, 11, 18)}), frozenset({(16, 13, 5), (5, 11, 14), (9, 13, 16), (14, 11, 9), (16, 14, 9), (9, 11, 13), (5, 14, 16), (13, 11, 5)}), frozenset({(15, 5, 3), (3, 9, 15), (18, 11, 5), (5, 15, 18), (18, 15, 9), (3, 5, 11), (11, 9, 3), (9, 11, 18)})}
the number of voids in each lags are 7 7 8 8 

Lag 3, 4에서 발견되고 1, 2에서 발견되지 않는 void : {fr

In [36]:
dim = 1
h1_t_persistence_1 = homol_rBD_from_pkl('h1_rBD_t_notes1_1e-4_0.0~1.5.pkl', dir="./pickle")
h1_t_persistence_2 = homol_rBD_from_pkl('h1_rBD_t_notes2_1e-4_0.0~1.5.pkl', dir="./pickle")
h1_t_persistence_3 = homol_rBD_from_pkl('h1_rBD_t_notes3_1e-4_0.0~1.5.pkl', dir="./pickle")
h1_t_persistence_4 = homol_rBD_from_pkl('h1_rBD_t_notes4_1e-4_0.0~1.5.pkl', dir="./pickle")
h1_4_t_1, h1_4_t_2, h1_4_t_3, h1_4_t_4, h1_4_t_all, h1_4_t_u = catch_intersection(h1_t_persistence_1, h1_t_persistence_2, h1_t_persistence_3, h1_t_persistence_4, dim)

Lag 1, 2, 3, 4 모두에서 발견되는 cycle : {(2, 9, 12, 5, 3), (0, 6, 9, 1), (0, 13, 8, 15), (5, 10, 9, 11), (3, 9, 16, 5), (0, 13, 9, 14, 16), (5, 13, 9, 11), (0, 19, 22, 1), (0, 15, 8, 18), (2, 9, 10, 5, 3), (0, 13, 9, 14, 18, 5), (0, 15, 22, 1), (0, 6, 9, 10, 5), (0, 5, 18, 19, 22, 1), (0, 6, 17, 1), (0, 5, 10, 9, 1), (5, 12, 9, 10), (5, 6, 9, 16), (0, 1, 9, 11, 5), (5, 10, 9, 13), (2, 9, 16, 5, 3), (5, 11, 9, 18, 19), (2, 9, 13, 5, 3), (5, 16, 14, 18), (2, 3, 5, 6, 9), (2, 20, 17, 6, 5, 18), (0, 1, 2, 18, 5), (3, 9, 10, 5), (5, 6, 9, 15), (5, 10, 9, 12), (5, 10, 9, 18), (5, 11, 9, 13), (5, 11, 9, 10), (2, 9, 11, 5, 3), (8, 13, 9, 15)}
the number of cycles in each lags are 48 49 48 44 

Lag 1에서만 발견되는 cycle : {(1, 9, 18, 5, 15, 22), (5, 18, 9, 10), (0, 1, 9, 18, 5), (1, 9, 18, 19, 22)}
Lag 2에서만 발견되는 cycle : {(2, 3, 5, 9)}
Lag 4에서만 발견되는 cycle : {(2, 9, 15, 5, 3)}


Lag 1, 2에서 발견되고 3, 4에서 발견되지 않는 cycle : {(3, 9, 14, 16, 5), (5, 6, 9, 12)}
Lag 1, 4에서 발견되고 2, 3에서 발견되지 않는 cycle : {(5, 6, 9, 18, 8, 1

In [38]:
dim = 1
h1_c_persistence_1 = homol_rBD_from_pkl('h1_rBD_c_notes1_1e-2_0.0~0.5_t0.6_s0.6.pkl', dir="./pickle")
h1_c_persistence_2 = homol_rBD_from_pkl('h1_rBD_c_notes2_1e-2_0.0~0.5_t0.6_s0.6.pkl', dir="./pickle")
h1_c_persistence_3 = homol_rBD_from_pkl('h1_rBD_c_notes3_1e-2_0.0~0.5_t0.6_s0.6.pkl', dir="./pickle")
h1_c_persistence_4 = homol_rBD_from_pkl('h1_rBD_c_notes4_1e-2_0.0~0.5_t0.6_s0.6.pkl', dir="./pickle")
h1_2_c_1, h1_2_c_2, h1_2_c_3, h1_2_c_4, h1_2_c_all, h1_2_c_u = catch_intersection(h1_c_persistence_1, h1_c_persistence_2, h1_c_persistence_3, h1_c_persistence_4, dim)

Lag 1, 2, 3, 4 모두에서 발견되는 cycle : {(5, 19, 9, 11), (5, 6, 9, 11), (0, 6, 9, 1), (3, 19, 5, 6, 9), (3, 9, 15, 5, 19), (0, 19, 22, 1), (5, 13, 9, 11), (0, 15, 8, 18), (5, 11, 9, 15), (0, 15, 22, 1), (0, 6, 17, 1), (3, 9, 11, 5), (0, 1, 9, 11, 5), (5, 6, 9, 19), (5, 11, 9, 18, 19), (5, 16, 14, 18), (0, 5, 6, 9, 1), (2, 20, 17, 6, 5, 18), (3, 9, 13, 5), (0, 19, 22, 15), (5, 18, 9, 11), (5, 19, 9, 15), (3, 9, 19, 5), (3, 9, 10, 5), (2, 9, 11, 5, 3), (5, 19, 18, 9, 11)}
the number of cycles in each lags are 26 26 26 26 

Lag 1, 2, 3, 4 모두에서 발견되는 26개의 cycle : {(5, 19, 9, 11), (5, 6, 9, 11), (0, 6, 9, 1), (3, 19, 5, 6, 9), (3, 9, 15, 5, 19), (0, 19, 22, 1), (5, 13, 9, 11), (0, 15, 8, 18), (5, 11, 9, 15), (0, 15, 22, 1), (0, 6, 17, 1), (3, 9, 11, 5), (0, 1, 9, 11, 5), (5, 6, 9, 19), (5, 11, 9, 18, 19), (5, 16, 14, 18), (0, 5, 6, 9, 1), (2, 20, 17, 6, 5, 18), (3, 9, 13, 5), (0, 19, 22, 15), (5, 18, 9, 11), (5, 19, 9, 15), (3, 9, 19, 5), (3, 9, 10, 5), (2, 9, 11, 5, 3), (5, 19, 18, 9, 11)}


In [126]:
# # 모든 lag에서 3개만 나오며, 위의 h2_t_2_all에 속해서 주석처리.
# # {frozenset({(15, 5, 3), (3, 9, 15), (18, 11, 5), (5, 15, 18), (18, 15, 9), (3, 5, 11), (11, 9, 3), (9, 11, 18)}), 
# # frozenset({(5, 11, 18), (18, 11, 9), (19, 18, 9), (3, 5, 19), (19, 9, 3), (3, 9, 11), (5, 18, 19), (11, 5, 3)}), 
# # frozenset({(3, 9, 19), (18, 11, 5), (9, 18, 19), (19, 5, 3), (19, 18, 5), (3, 5, 11), (11, 9, 3), (9, 11, 18)})}

# dim = 2
# h2_c_persistence_1 = homol_rBD_from_pkl('h2_rBD_c_notes1_1e-2_0.0~0.5_t0.6_s0.6.pkl', dir="./pickle")
# h2_c_persistence_2 = homol_rBD_from_pkl('h2_rBD_c_notes2_1e-2_0.0~0.5_t0.6_s0.6.pkl', dir="./pickle")
# h2_c_persistence_3 = homol_rBD_from_pkl('h2_rBD_c_notes3_1e-2_0.0~0.5_t0.6_s0.6.pkl', dir="./pickle")
# h2_c_persistence_4 = homol_rBD_from_pkl('h2_rBD_c_notes4_1e-2_0.0~0.5_t0.6_s0.6.pkl', dir="./pickle")
# h2_2_c_1, h2_2_c_2, h2_2_c_3, h2_2_c_4, h2_2_c_all, h2_2_c_u = catch_intersection(h2_c_persistence_1, h2_c_persistence_2, h2_c_persistence_3, h2_c_persistence_4, dim)


In [127]:
dim = 2
h2_c_persistence_1 = homol_rBD_from_pkl('h2_rBD_c_notes1_1e-2_0.0~0.5_t0.3_s0.6.pkl', dir="./pickle")
h2_c_persistence_2 = homol_rBD_from_pkl('h2_rBD_c_notes2_1e-2_0.0~0.5_t0.3_s0.6.pkl', dir="./pickle")
h2_c_persistence_3 = homol_rBD_from_pkl('h2_rBD_c_notes3_1e-2_0.0~0.5_t0.3_s0.6.pkl', dir="./pickle")
h2_c_persistence_4 = homol_rBD_from_pkl('h2_rBD_c_notes4_1e-2_0.0~0.5_t0.3_s0.6.pkl', dir="./pickle")
h2_2_c_1, h2_2_c_2, h2_2_c_3, h2_2_c_4, h2_2_c_all, h2_2_c_u = catch_intersection(h2_c_persistence_1, h2_c_persistence_2, h2_c_persistence_3, h2_c_persistence_4, dim)

Lag 1, 2, 3, 4 모두에서 발견되는 void : {frozenset({(3, 9, 19), (18, 11, 5), (9, 18, 19), (19, 5, 3), (19, 18, 5), (3, 5, 11), (11, 9, 3), (9, 11, 18)}), frozenset({(15, 9, 3), (15, 5, 0), (19, 18, 0), (0, 5, 19), (3, 9, 19), (18, 15, 0), (9, 15, 18), (9, 18, 19), (3, 5, 15), (19, 5, 3)}), frozenset({(16, 13, 5), (5, 11, 14), (9, 13, 16), (14, 11, 9), (16, 14, 9), (9, 11, 13), (5, 14, 16), (13, 11, 5)}), frozenset({(15, 5, 3), (3, 9, 15), (18, 11, 5), (5, 15, 18), (18, 15, 9), (3, 5, 11), (11, 9, 3), (9, 11, 18)}), frozenset({(15, 5, 0), (9, 15, 19), (11, 17, 18), (0, 1, 19), (17, 6, 5), (0, 5, 6), (19, 15, 0), (17, 9, 1), (18, 17, 15), (1, 9, 19), (18, 15, 9), (17, 11, 9), (5, 15, 17), (6, 1, 0), (1, 6, 17), (9, 11, 18)})}
the number of voids in each lags are 5 5 5 5 

Lag 1, 2, 3, 4 모두에서 발견되는 5개의 void : {frozenset({(3, 9, 19), (18, 11, 5), (9, 18, 19), (19, 5, 3), (19, 18, 5), (3, 5, 11), (11, 9, 3), (9, 11, 18)}), frozenset({(15, 9, 3), (15, 5, 0), (19, 18, 0), (0, 5, 19), (3, 9, 19), (18, 

In [100]:
h1_s_persistence = homol_rBD_from_pkl('h1_rBD_s_notes_1e-2.pkl', dir="./pickle")

In [101]:
h1_2_s = []
for cycle in h1_s_persistence.keys():
    h1_2_s.append(cycle)
h1_2_s = set(h1_2_s)

h1_2_s

{(2, 9, 3, 14, 18, 5, 17),
 (3, 14, 18, 5, 20, 9),
 (3, 19, 15, 5, 18, 14),
 (5, 14, 9, 17),
 (5, 14, 9, 19),
 (5, 14, 9, 20)}

* search for relation between h1(cycle) and h2(void) with degree sequence [4, 4, 4, 4, 4, 4]

In [139]:
v1 = 0
v2 = 9
cycles_in_any_lag = h1_4_t_u
cycles_in_all_lag = h1_4_t_all

count = 0
count_all = 0
for cycle in cycles_in_any_lag :
    if (v1 in cycle) and (v2 in cycle) :
        print(cycle)
        if_or_not = cycle in cycles_in_all_lag
        print(f"it is {if_or_not} that it appear in search for all lags(1, 2, 3, 4)")
        count += 1
        count_all += if_or_not
        # cycles_in_all_lag = cycles_in_all_lag - {cycle}

print(count, count_all) 

(0, 6, 9, 1)
it is True that it appear in search for all lags(1, 2, 3, 4)
(0, 13, 9, 14, 16)
it is True that it appear in search for all lags(1, 2, 3, 4)
(0, 1, 9, 18, 5)
it is False that it appear in search for all lags(1, 2, 3, 4)
(0, 13, 9, 14, 18, 5)
it is True that it appear in search for all lags(1, 2, 3, 4)
(0, 6, 9, 10, 5)
it is True that it appear in search for all lags(1, 2, 3, 4)
(0, 5, 18, 14, 9, 13)
it is False that it appear in search for all lags(1, 2, 3, 4)
(0, 13, 9, 14, 16, 5)
it is False that it appear in search for all lags(1, 2, 3, 4)
(0, 5, 10, 9, 1)
it is True that it appear in search for all lags(1, 2, 3, 4)
(0, 1, 9, 11, 5)
it is True that it appear in search for all lags(1, 2, 3, 4)
9 6


In [50]:
len(cycles_in_any_lag), len(cycles_in_all_lag)

(57, 35)

In [89]:
cycle_to_check = (5, 16, 9, 14, 18)

for cycle_in_a_lg in [h1_4_t_1, h1_4_t_2, h1_4_t_3, h1_4_t_4] :
    print(cycle_to_check in cycle_in_a_lg)

False
False
True
True


* 어떤 화음에 속한 노드 중 2개 이상이 timeflow cycle과 complex cycle을 구성하는 경우를 추적하는 함수

In [ ]:
# import util
# import importlib
# importlib.reload(util)
from util import find_cycles_with_simul_intersection

result_cycles, result_keys = find_cycles_with_simul_intersection(h1_2_s, notes_dict)

# 결과 출력
for i, cycle in enumerate(result_cycles[:5]):
    print(f"Cycle: {cycle}, Chord Keys: {result_keys[i]}")

    h1_2_c_u

Cycle: (5, 14, 9, 17), Chord Keys: [11]
Cycle: (5, 14, 9, 20), Chord Keys: [10]
Cycle: (5, 14, 9, 19), Chord Keys: [1, 15]
Cycle: (3, 19, 15, 5, 18, 14), Chord Keys: [4, 6, 15]
Cycle: (3, 14, 18, 5, 20, 9), Chord Keys: [4, 10]


* 아무래도 complex cycle이 좀 더 같은 화음에 있는 노드들로 구성될 확률이 높다.(탐색한 rate가 0.01 vs 0.0001로 다르긴 하지만...)

In [158]:
result_cycles_c, result_keys_c = find_cycles_with_simul_intersection(h1_2_c_u, notes_dict)

kinda_weight_c = 0
for i, cycle in enumerate(result_cycles_c):
    # print(f"Cycle: {cycle}, Chord Keys: {result_keys_c[i]}")
    kinda_weight_c += len(result_keys_c[i])
print(len(result_cycles_c), len(h1_2_c_u), len(result_cycles_c) / len(h1_2_c_u), kinda_weight_c / len(result_cycles_c))

result_cycles_t, result_keys_t = find_cycles_with_simul_intersection(h1_4_t_u, notes_dict)

kinda_weight_t = 0
for i, cycle in enumerate(result_cycles_t):
    # print(f"Cycle: {cycle}, Chord Keys: {result_keys_t[i]}")
    kinda_weight_t += len(result_keys_t[i])
print(len(result_cycles_t), len(h1_4_t_u), len(result_cycles_t) / len(h1_4_t_u), kinda_weight_t / len(result_cycles_t))

24 26 0.9230769230769231 2.5833333333333335
44 57 0.7719298245614035 2.25


In [154]:
h1_2_c_u - h1_4_t_u

{(0, 5, 6, 9, 1),
 (0, 19, 22, 15),
 (3, 9, 11, 5),
 (3, 9, 15, 5, 19),
 (3, 9, 19, 5),
 (3, 19, 5, 6, 9),
 (5, 6, 9, 11),
 (5, 6, 9, 19),
 (5, 18, 9, 11),
 (5, 19, 9, 11),
 (5, 19, 9, 15),
 (5, 19, 18, 9, 11)}

* 손으로 그리는 데 도움받는 용도...

In [136]:
from util import get_degree_sequence

temp = h2_2_c_u - h2_t_2_u

for h2 in temp:
    temp2 = set(h2)
    print("2-simplex building 2-homology :", temp2)
    deg_seq = get_degree_sequence(temp2)
    print("with", len(deg_seq), "nodes and degree sequence :", deg_seq, "\n")

2-simplex building 2-homology : {(15, 5, 0), (17, 6, 5), (17, 9, 1), (18, 15, 9), (17, 11, 9), (6, 1, 0), (9, 11, 18), (9, 15, 19), (11, 17, 18), (0, 1, 19), (0, 5, 6), (19, 15, 0), (18, 17, 15), (1, 9, 19), (5, 15, 17), (1, 6, 17)}
with 10 nodes and degree sequence : {15: 6, 5: 4, 0: 5, 17: 7, 6: 4, 9: 6, 1: 5, 18: 4, 11: 3, 19: 4} 

2-simplex building 2-homology : {(15, 9, 3), (15, 5, 0), (19, 18, 0), (0, 5, 19), (3, 9, 19), (18, 15, 0), (9, 15, 18), (9, 18, 19), (3, 5, 15), (19, 5, 3)}
with 7 nodes and degree sequence : {15: 5, 9: 4, 3: 4, 5: 4, 0: 4, 19: 5, 18: 4} 



In [ ]:
def get_brief_homol_rBD(homology_persistence : dict) :

    desired_dict = {}
    for homol in homology_persistence.keys():
        homol_ends = [homology_persistence[homol][0], homology_persistence[homol][-1]]
        birth_rate = homol_ends[0][0]
        death_rate = homol_ends[1][0]
        # rate_persistence = homol_ends[1][0] - homol_ends[0][0] # rate persistence
        life_at_birth_rate = homol_ends[0][2] - homol_ends[0][1] # life at birth rate
        life_at_death_rate = homol_ends[1][2] - homol_ends[1][1] # life at death rate
        desired_dict[homol] = [birth_rate, death_rate, life_at_birth_rate, life_at_death_rate]

    return desired_dict

brief_1 = get_brief_homol_rBD(h1_t_persistence_1) 
brief_1[(2, 20, 17, 6, 5, 18)]

[0.0, 1.4999, 0.007692307692307693, 0.002806894202297754]

* inter_weight로만 실행했을 때

In [ ]:
# from util import get_chords_inter_connected, is_distance_matrix_from, analyze_lifespans
# from professor import generateBarcode

# inter_lag = 1
# inter_weight = get_chords_inter_connected(adn_i[1][inter_lag], adn_i[2][inter_lag], lag=inter_lag)

# oor_inter = 1 + 2 * 1 / inter_weight[inter_weight!=0].min().min() # 2
# inter_distance = is_distance_matrix_from(inter_weight, notes_dict, out_of_reach = oor_inter)

# birthDeath_inter = generateBarcode(mat = inter_distance.values, exactStep = True, birthDeathSimplex=False, sortDimension=False)
# cycle_profile_inter = [(1e+6, birthDeath_inter)]
# cycles_info_inter = analyze_lifespans(cycle_profile_inter, out_of_reach = 1)
# cycle_persistence_inter = get_rBD_groupedBy_homol(cycle_profile_inter)

# for cycle, rBD in cycle_persistence_inter.items():
#     for rate, birth, death in rBD :
#         print(f"{cycle} : {round(birth, 6)} - {round(death, 6)}")

In [ ]:
# simul_cycles = []
# for simul_cycle in cycle_persistence_inter.keys():
#     simul_cycles.append(simul_cycle)

# cycle_persistence_t_n1 = get_rBD_groupedBy_homol(cycles_profile_t_n1)
# timeflow_cycles_n1 = []
# for cycle in cycle_persistence_t_n1.keys():
#     timeflow_cycles_n1.append(cycle)

# set(timeflow_cycles_n1).intersection(set(simul_cycles))

* 생각보다 적게 나온다.

* cycle_profile에 저장되어있는 정보를 딕셔너리로 정제합니다.
* key : 사이클을 구성하는 vertex들을 가장 작은 정수로 레이블링된 것부터 연결된 순서대로 나열
* value : 해당 사이클이 발견된 시점에 대해 (rate, birth, death) tuple를 담은 리스트

### complex cycle / void 탐색


* timeflow cycle 개수는 대략 0.5, 0.65, 0.95를 기점으로 계단이 나뉘었다.

In [ ]:
from util import (search_complex_homology, get_rBD_groupedBy_homol, 
                  split_cycles_by_consecutive, plot_homol_BirthDeath_over_rate, 
                  homol_rBD_to_pkl)

rate_t = 0.3
rate_s = 0.6

start_c = 0.0
end_c = 0.5
power_c = -2


In [ ]:
for dim_c in [1, 2] :
    for inter_lag_t in [1, 2, 3, 4] :

        homol_profile, outta_reach = search_complex_homology(adn_i, inter_lag_t, notes_dict, rate_t, rate_s, 
                                                            dim_c, start_c, end_c, power_c)

        homol_persistence_c = get_rBD_groupedBy_homol(homol_profile, dim = dim_c)
        print("Total number of homology: ", len(homol_persistence_c))

        non_consecutive_homol_s, consecutive_homol_s = split_cycles_by_consecutive(homol_persistence_c, out_of_reach = outta_reach, power = power_c)

        if len(non_consecutive_homol_s) >= 1 :
            plot_homol_BirthDeath_over_rate(non_consecutive_homol_s, subplot_in_a_row = 4, power = power_c)
            homol_persistence_c_updated = {**non_consecutive_homol_s, **consecutive_homol_s}
            print("Hmm.. you need to check this")
        else :
            homol_persistence_c_updated = consecutive_homol_s

        plot_homol_BirthDeath_over_rate(consecutive_homol_s, subplot_in_a_row = 4, power = power_c)


        _ = homol_rBD_to_pkl(homol_persistence_c, notes_dict, inter_lag_t, dim_c, start_c, end_c, power_c, 
                            type = 'c', rate_t = rate_t, rate_s = rate_s)

### 2번 꺾이는 게 있네... 신기하다

* 악보로 확인하는 것까지만 필요할 때 (evaluate_threshold 세부 기능 중)

In [7]:
simul_whole_c = simul_chord_lists(adn_i[1][-1], adn_i[2][-1])
simul_notes = simul_union_by_dict(simul_whole_c, notes_dict)
nodes_list = list(range(1, 23+1))
hibari_notes_df = get_correct_df(nodes_list, simul_notes)

In [8]:
threshold = 0.25
focus = [9] # 모든 사이클을 다 악보로 확인하려면 None, 특정 사이클만 보고 싶으면 cycle_labeled에서 인덱스 확인해서 list로 전달 e.g. [10, 13]
h1_c_persistence_1 = homol_rBD_from_pkl('h1_rBD_c_notes1_1e-2_0.0~0.5_t0.6_s0.6.pkl', dir="./pickle")

In [9]:
cycle_labeled, length_counts_in_c, note_counts_in_c = label_cycle(h1_c_persistence_1, notes_dict, log = False)
h1_c_cycles_weak = get_scattered_cycles_df(df = hibari_notes_df, cycle_labeled = cycle_labeled, binary = True)

for_scores = get_scores_for_cycle_scaled(h1_c_cycles_weak, cycle_labeled, threshold, lower_bound = None, indices_2_check = focus)

flexible_pitches = get_flexible_pitches(notes_counts, notes_label, log = False)
hibari_lists = [adn_1_real, adn_2_real]


output_dir = f'./test_xml/{get_now()}'
verify_cycles_scaled_by_scores(hibari_lists, cycle_labeled, for_scores, focus, notes_label, flexible_pitches, 
                               output_dir = output_dir, type = 'c')

cycle of index 9 :
MusicXML 파일이 저장되었습니다: ./test_xml/20250518_171601\c9. [3, 21, 18, 7, 6, 19] on scale 31.musicxml


[[[(208, 59, 210),
   (209, 72, 210),
   (210, 60, 212),
   (212, 71, 213),
   (212, 55, 214),
   (213, 76, 214),
   (214, 72, 215),
   (215, 76, 216),
   (216, 59, 218),
   (217, 71, 218),
   (218, 72, 219),
   (224, 59, 226),
   (225, 72, 226),
   (226, 60, 228),
   (228, 71, 229),
   (228, 55, 230),
   (229, 76, 230),
   (230, 72, 231),
   (232, 76, 233),
   (232, 59, 234),
   (233, 72, 234),
   (296, 76, 297),
   (296, 59, 298),
   (297, 72, 298),
   (304, 59, 306),
   (305, 72, 306),
   (306, 60, 308),
   (308, 71, 309),
   (308, 55, 310),
   (309, 76, 310),
   (310, 72, 311),
   (311, 76, 312),
   (312, 59, 314),
   (313, 71, 314),
   (314, 72, 315),
   (320, 59, 322),
   (321, 72, 322),
   (322, 60, 324),
   (324, 71, 325),
   (324, 55, 326),
   (325, 76, 326),
   (326, 72, 327),
   (328, 76, 329),
   (328, 59, 330),
   (329, 72, 330),
   (336, 59, 338),
   (337, 72, 338),
   (338, 60, 340),
   (340, 71, 341),
   (340, 55, 342),
   (341, 76, 342),
   (342, 72, 343),
   (343, 76,

* 중첩행렬을 건설하는 것까지만 필요할 때 (evaluate_threshold 세부 기능 중)

In [66]:
# from util import get_cycles_scaled, construct_overlap_df
# from process import analyze_scale_reduction

# goal = 0.6
# for_overlap = get_cycles_scaled(cycles_weak, cycle_labeled, goal, lower_bound = None)
# overlapped_cycles = construct_overlap_df(for_overlap)
# actual_on = analyze_scale_reduction(cycles_weak, overlapped_cycles, goal)

# from util import get_now
# from professor import plot_OM

# timestamp = get_now()
# songname = f'complexOM_hibari_th={goal}_actual_{round(actual_on, 2)}'
# plot_OM(overlapped_cycles, songname)

## 이런 식으로 나온 것 중에 timeflow에서 뽑은 2-simplex와 겹치는 게 있을 수도 있지 않을까?

### simul cycle / void 탐색
  * void는 아예 없다.

In [56]:
# from util import search_simul_cycles

# power_s = -2
# dim_s = 1
# start_s = 0.0
# end_s = 1.5

In [57]:
# homol_profile_s, outta_reach_s = search_simul_cycles(adn_i, notes_dict, dimension = dim_s, power = power_s)

# h2_persistence_s = get_rBD_groupedBy_homol(homol_profile_s, dim = dim_s)

# non_consecutive_h2_s, consecutive_h2_s = split_cycles_by_consecutive(h2_persistence_s, out_of_reach = outta_reach_s, power = power_s)
# plot_homol_BirthDeath_over_rate(consecutive_h2_s, subplot_in_a_row = 3)

# if len(non_consecutive_h2_s) >= 1 :
#     h2_persistence_s_updated = {**non_consecutive_h2_s, **consecutive_h2_s}
#     print("Hmm.. you need to check this")
# else :
#     h2_persistence_s_updated = consecutive_h2_s

# # 데이터프레임 변환 및 pkl파일 내보내기
# temp_s_n_5 = homol_rBD_to_pkl(h2_persistence_s_updated, notes_dict, None, dim_s, start_s, end_s, power_s, type = 's')

* 악보 전체 범위에 대한 Chord labelled list (simul_whole_c)를 만들고, notes_dict를 이용해 note labelling을 해줍니다.
* inst 1은 마지막 4마디에 쉬고, inst 2는 처음 4마디를 쉽니다. 

In [ ]:
from process import simul_chord_lists
from util import simul_union_by_dict

simul_whole_c = simul_chord_lists(adn_i[1][0], adn_i[2][0])
simul_notes = simul_union_by_dict(simul_whole_c, notes_dict)

print("At First bar, inst 1 playes solo", simul_notes[0:5])
print("At Fifth bar, inst 1 & 2 play   ", simul_notes[32:37])

# print(simul_notes[:32] == simul_notes[-32:]) # True (firth 4 and last 4 bars are played solo)
# print(simul_notes[:33] == simul_notes[-33:]) # False

At First bar, inst 1 playes solo [{9, 15, 6, 1}, {9, 19, 6, 1}, {16, 2, 20, 7}, {2, 23, 7}, {10, 18, 3, 22}]
At Fifth bar, inst 1 & 2 play    [{9, 15, 6, 1}, {1, 6, 9, 15, 19}, {1, 2, 6, 7, 9, 16, 19, 20}, {16, 2, 7, 20, 23}, {18, 3, 2, 7, 22, 23, 10}]


In [ ]:
# from util import get_correct_df, get_scattered_cycles_df

# simul_notes = simul_union_by_dict(simul_whole_c, notes_dict)
# nodes_list = list(range(1, 23+1))
# hibari_notes_df = get_correct_df(nodes_list, simul_notes)
# cycles_weak = get_scattered_cycles_df(df = hibari_notes_df, cycle_labeled = cycle_labeled, weak = True)

# # 위에도 있는데 너무 멀어서 다시 가져옴
# module_notes = adn_1_real[:59]
# active_module = group_notes_with_duration_(module_notes)
# notes_label, notes_counts = notes_label_n_counts(module_notes)

# from util import notes_label_counts, analyze_cycles_scattered #, check_ratio_of_on

# notes_freq = notes_label_counts(notes_label, notes_counts)
# plausible = analyze_cycles_scattered(cycles_weak, cycle_labeled, notes_freq, scale = True)
# plausible[:5] # (column_idx, ratio, cycle_length, average_freq)

In [31]:
# notes_freq_ = sorted(notes_freq.items(), key=lambda item: item[1])
# notes_freq_